# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jayanthGowda1718/ml-internship-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Using the feature set defined in the data contract (w03_data_contract), we build a clean feature matrix: numeric features used as-is, categorical features one-hot encoded, and missing values filled with sensible defaults rather than dropped, to preserve row count.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/jayanthGowda1718/ml-internship-work/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "sessions_90d", "users_90d",
    "content_age_days", "avg_position"
]

categorical_features = [
    "content_type", "main_intent", "freshness_tier", "word_count_tier"
]

# Fill missing numeric values with median (robust to outliers)
for col in numeric_features:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with an explicit "unknown" label
for col in categorical_features:
    df[col] = df[col].fillna("unknown")

# One-hot encode categoricals
X = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features)

print(X.shape)
X.head()

(30000, 27)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,sessions_90d,users_90d,content_age_days,avg_position,...,main_intent_unknown,freshness_tier_0-30,freshness_tier_181+,freshness_tier_31-90,freshness_tier_91-180,word_count_tier_1000-2000,word_count_tier_2000-3500,word_count_tier_3500+,word_count_tier_<1000,word_count_tier_unknown
0,10.0,0.67,2.05,3221.0,20457.0,3803,17,16,187,10.6,...,False,True,False,False,False,False,True,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,15320,9,9,445,20.3,...,False,True,False,False,False,False,True,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,11,141,36.5,...,False,True,False,False,False,False,False,True,False,False
3,10.0,0.00,0.00,2877.0,19116.0,11751,78,75,463,6.2,...,False,True,False,False,False,False,False,False,False,True
4,0.0,0.00,0.00,2803.0,17469.0,19140,145,144,263,44.0,...,False,True,False,False,False,False,True,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- search_volume: monthly search demand for the page's target query. No missing values expected; available before prediction (it's query-level, not outcome-level).
- competition / competition_level: how contested the keyword is. Numeric/categorical pair; available before prediction.
- cpc: cost-per-click benchmark for the query, a proxy for commercial value. Available before prediction.
- content_type: page format (e.g. blog, product, guide). Categorical, filled with "unknown" if missing; available before prediction — it's a property of the page itself.
- main_intent: search intent category. Categorical, same handling; available before prediction.
- word_count / char_count: length of the content. Numeric, filled with median if missing; available before prediction.
- impressions_90d, sessions_90d, users_90d: trailing 90-day traffic volume. These exist before the *current* prediction moment since they're historical, but must never include the same period as the label window to avoid leakage.
- content_age_days: how old the page is. Available before prediction, a pure timestamp-derived fact.
- avg_position: average search rank over the 90-day window. Available before prediction as a historical fact, but excluded from being confused with the *future* position we'd want to improve.
- freshness_tier, word_count_tier: pre-computed bucket versions of content_age_days and word_count. Included as categorical conveniences; available before prediction since they're derived from static page properties.

Excluded from features (leakage risk — see w03_data_contract): ctr, engagement_rate, clicks_90d, engaged_sessions_90d, scroll_events_90d, scroll_rate, position_tier, impression_tier, trend_direction, trend_pct — all either define or are directly derived from the label itself.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking my own feature set for three leakage patterns: (1) columns directly derived from the label, (2) columns that peek into a future window relative to prediction time, (3) pre-computed product flags/tiers that encode the answer.

In [4]:

label = df["ctr"]

suspects = ["clicks_90d", "engaged_sessions_90d", "scroll_events_90d", "scroll_rate",
            "position_tier", "impression_tier", "trend_direction", "trend_pct"]

for col in suspects:
    if df[col].dtype in ["int64", "float64"]:
        corr = df[col].corr(label)
        print(f"{col}: correlation with ctr = {corr:.3f}")
    else:
        print(f"{col}: categorical, skipped numeric correlation — inspect manually")

clicks_90d: correlation with ctr = 0.011
engaged_sessions_90d: correlation with ctr = 0.009
scroll_events_90d: correlation with ctr = 0.002
scroll_rate: correlation with ctr = 0.013
position_tier: categorical, skipped numeric correlation — inspect manually
impression_tier: categorical, skipped numeric correlation — inspect manually
trend_direction: categorical, skipped numeric correlation — inspect manually
trend_pct: correlation with ctr = 0.008


In [2]:

check = df["clicks_90d"] / df["impressions_90d"]
print(check.corr(df["ctr"]))

0.9999997838677429


In [3]:

print(df["impressions_last_30d"].sum(), df["impressions_90d"].sum())


42871762 156010989


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- clicks_90d: excluded — mechanically defines ctr (ctr = clicks_90d / impressions_90d), confirmed by near-1.0 correlation in the leakage hunt.
- engaged_sessions_90d: excluded — directly used to compute engagement_rate, one of our label sources.
- scroll_events_90d, scroll_rate: excluded — scroll_rate is derived from scroll_events_90d and both are downstream engagement signals correlated with the label.
- position_tier, impression_tier: excluded — pre-computed bucket versions of avg_position and impressions_90d that may encode outcome-adjacent groupings rather than raw, independent signal.
- trend_direction, trend_pct: excluded — these compare last_30d vs prev_30d performance, which is itself a summary of the change we're trying to predict, not an independent predictor.
- ai_sessions_90d, ai_traffic_pct: excluded — belongs to a different lane (ai_opportunity), not relevant to engagement_fix and risks mixing unrelated signal.
- provider_used, model_used: excluded — internal content-production tooling metadata, not a signal about page performance.
- client_id: excluded as a feature (kept only as context) — using it directly could let the model memorize client-specific quirks rather than learning generalizable patterns.

In [5]:
excluded = [
    "clicks_90d", "engaged_sessions_90d", "scroll_events_90d", "scroll_rate",
    "position_tier", "impression_tier", "trend_direction", "trend_pct",
    "ai_sessions_90d", "ai_traffic_pct", "provider_used", "model_used", "client_id"
]

still_present = [col for col in excluded if any(col in c for c in X.columns)]
print("Excluded columns still present in X:", still_present)

Excluded columns still present in X: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.